# QA Agent — user perspective

**What it does.** Reviews a PR diff against the linked Linear work item across 7 dimensions (functional / AC / quality / architecture / regression / edges / tests). Returns at most 5 findings (QAAG-03).

**Hard cycle cap.** `qa_cycle_count` is bounded at 3. The Pydantic `model_validator` on `QAOutput` rejects `qa_cycle_count >= 3` AND `qa_status="changes_required"` — last line of defence against QA runaway (QAAG-04, Pitfall 2).

**Entry point.** `run_qa_agent(input: QAInput) -> QAOutput`. After validation succeeds, `_write_qa_results_to_linear` increments the cycle counter and creates fix subtasks (post-loop, never inside the agent).

**Cost.** Live runs read the PR via `gh pr diff/view` and call Linear after. Gated.

In [ ]:
import sys
from pathlib import Path

# Robust _helpers.py discovery — works whether jupyter lab was launched from
# the repo root, notebooks/, or notebooks/agents/. The loop walks up from cwd
# until it finds the parent that holds _helpers.py and inserts it on sys.path.
nb_dir = Path.cwd()
for _parent in (nb_dir, *nb_dir.parents):
    if (_parent / "_helpers.py").exists():
        sys.path.insert(0, str(_parent))
        break

from _helpers import ensure_src_on_path, runtime_summary

ROOT = ensure_src_on_path()
print("HSB_RUNTIME_<AGENT> selection:\n" + runtime_summary())

## Construct the input

`qa_cycle_count` is **0-indexed in the input** (`0`=first review, `1`=second, `2`=third). The output is **1-indexed** (the agent increments). Pass the full PR diff text — the agent has no `gh` write privileges, only read.

In [ ]:
from hsb.contracts.qa import PullRequestInput, QAInput

example_input = QAInput(
    work_item_id="LIN-456",
    linear_issue={
        "id": "LIN-456",
        "title": "Add docstring to greet()",
        "description": "Improve readability.",
        "acceptance_criteria": ["greet() has a docstring"],
    },
    pull_request=PullRequestInput(
        url="https://github.com/example/repo/pull/42",
        diff=(
            "diff --git a/scratch.py b/scratch.py\n"
            "+def greet(name):\n"
            "+    # Return a greeting for ``name``.\n"
            "+    return f'hello {name}'\n"
        ),
    ),
    qa_cycle_count=0,  # first review
)
print(example_input.model_dump_json(indent=2))

## Live invocation

Gated on `HSB_NOTEBOOK_RUN_LIVE=1`. The default `gh pr diff` URL above is a placeholder — replace with a real PR URL before running live.

In [ ]:
from _helpers import assert_g1_safe, gated, live_mode

if not live_mode():
    print(gated("QA Agent live run"))
else:
    assert_g1_safe()
    from hsb.agents.qa_agent import run_qa_agent

    out = run_qa_agent(example_input)
    print("status:", out.qa_status)
    print("cycle:", out.qa_cycle_count, "/ 3")
    print("findings:", len(out.findings))
    for f in out.findings:
        print(f"  [{f.severity}] {f.title}")